# Small-scale adaptation: DDAIR

**Original paper:** Galo Castillo-López, Alexis Lombard, Nasredine Semmar, and Gaël de Chalendar, *How DDAIR you? Disambiguated Data Augmentation for Intent Recognition*, EACL 2026 Short Papers, pp. 274–286. Official record: https://aclanthology.org/2026.eacl-short.20/.

This adaptation tests the paper's disambiguation intuition with transparent local operations rather than its exact sentence-embedding pipeline. It makes one lexical augmentation per training utterance, then retains an augmented example only when its TF-IDF cosine similarity to the source class centroid exceeds the strongest competing centroid by a margin. No reproduction or claim about the paper's results is made.

In [1]:
import csv, io, json, random, re, urllib.request
from pathlib import Path
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier

SEED = 42
random.seed(SEED); np.random.seed(SEED)
DATA_URL = "https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/master/banking_data/train.csv"
TEST_URL = "https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/master/banking_data/test.csv"
CLASSES = ["card_arrival", "card_linking", "exchange_rate", "card_payment_wrong_exchange_rate", "extra_charge_on_statement", "pending_cash_withdrawal", "fiat_currency_support", "card_delivery_estimate", "automatic_top_up", "card_not_working"]
# Keep a compact, public, reproducible subset. If a category is absent in a future mirror, fail clearly.
def load_rows(url):
    raw = urllib.request.urlopen(url, timeout=30).read().decode("utf-8")
    return list(csv.DictReader(io.StringIO(raw)))
train_rows = [r for r in load_rows(DATA_URL) if r["category"] in CLASSES]
test_rows = [r for r in load_rows(TEST_URL) if r["category"] in CLASSES]
if set(r["category"] for r in train_rows) != set(CLASSES):
    raise RuntimeError("The public Banking77 mirror does not contain the expected compact class subset")
X_all = np.array([r["text"] for r in train_rows], dtype=object)
y_all = np.array([r["category"] for r in train_rows], dtype=object)
X_test = np.array([r["text"] for r in test_rows], dtype=object)
y_test = np.array([r["category"] for r in test_rows], dtype=object)
# The official test file is held out. The train file is split again for a deterministic validation-style adaptation.
X_train, X_dev, y_train, y_dev = train_test_split(X_all, y_all, test_size=0.25, random_state=SEED, stratify=y_all)

def model():
    return Pipeline([("tfidf", TfidfVectorizer(lowercase=True, ngram_range=(1,2), min_df=1, sublinear_tf=True)), ("clf", LogisticRegression(max_iter=400, random_state=SEED))])
def score(name, clf, Xtr, ytr, Xev=X_test, yev=y_test):
    clf.fit(Xtr, ytr); pred=clf.predict(Xev)
    return {"condition": name, "accuracy": round(float(accuracy_score(yev,pred)), 6), "macro_f1": round(float(f1_score(yev,pred,average="macro")), 6), "n_train": int(len(Xtr)), "n_test": int(len(yev))}

In [2]:
from sklearn.metrics.pairwise import cosine_similarity

def augment(text):
    swaps={"card":"bank card","fee":"charge","cash":"money","payment":"purchase","received":"got"}
    words=text.split(); out=[]
    for w in words:
        key=re.sub(r"[^a-z]","",w.lower())
        out.append(swaps.get(key,w))
    return " ".join(out)

X_aug=np.array([augment(x) for x in X_train],dtype=object)
vec=TfidfVectorizer(ngram_range=(1,2),min_df=1)
Z=vec.fit_transform(np.concatenate([X_train,X_aug]))
Zc=Z[:len(X_train)]; Za=Z[len(X_train):]
centroids={c:Zc[np.where(y_train==c)[0]].mean(axis=0) for c in CLASSES}
keep=[]; margins=[]
for i,c in enumerate(y_train):
    own=float(Za[i].dot(centroids[c].T).__array__()[0,0])
    others=[float(Za[i].dot(centroids[o].T).__array__()[0,0]) for o in CLASSES if o!=c]
    margin=own-max(others); margins.append(margin); keep.append(margin>=0.02)
keep=np.array(keep); margins=np.array(margins)
X_filtered=np.concatenate([X_train,X_aug[keep]]); y_filtered=np.concatenate([y_train,y_train[keep]])
results=[score("original training",model(),X_train,y_train),score("all lexical augmentations",model(),np.concatenate([X_train,X_aug]),np.concatenate([y_train,y_train])),score("margin-filtered augmentations",model(),X_filtered,y_filtered)]
out={"dataset":"Banking77 compact 10-class subset","augmentation":"deterministic lexical substitutions","margin_threshold":0.02,"kept_augmented":int(keep.sum()),"results":results}
Path("results.json").write_text(json.dumps(out,indent=2))
print(json.dumps(out,indent=2))

{
  "dataset": "Banking77 compact 10-class subset",
  "augmentation": "deterministic lexical substitutions",
  "margin_threshold": 0.02,
  "kept_augmented": 819,
  "results": [
    {
      "condition": "original training",
      "accuracy": 0.95,
      "macro_f1": 0.950371,
      "n_train": 1017,
      "n_test": 400
    },
    {
      "condition": "all lexical augmentations",
      "accuracy": 0.9525,
      "macro_f1": 0.952806,
      "n_train": 2034,
      "n_test": 400
    },
    {
      "condition": "margin-filtered augmentations",
      "accuracy": 0.9425,
      "macro_f1": 0.942836,
      "n_train": 1836,
      "n_test": 400
    }
  ]
}
